## Task One of the Detheon ML and DS Internship
### Abdul-Azeez Mahmud Gambo | mahmudabdulazeez9@gmail.com

##### Detailed Problem Statement & Requirements
Construct a natural language processing text classifier. Ingest customer product or movie review records, clean and tokenize text (lowercasing, stopword removal, lemmatization), transform text into numerical TF-IDF feature matrices, and train a Multinomial Naive Bayes classifier to identify positive and negative sentiment.

##### Suggested Technical Approach
1. Download the sentiment reviews dataset.
2. Preprocess text using NLTK or regex (remove HTML tags, punctuation, stopwords).
3. Convert text to numerical vectors using TfidfVectorizer(max_features=5000).
4. Train MultinomialNB and LogisticRegression.
5. Create a predict_sentiment() function that tests live custom user input phrases.

In [3]:
!pip install nltk

In [11]:
import nltk
import re
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [15]:
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...


True

In [16]:
df = pd.read_csv("sentiment_reviews.csv")

In [17]:
# Previewing the data information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   review_id    10 non-null     int64
 1   review_text  10 non-null     str  
 2   sentiment    10 non-null     str  
dtypes: int64(1), str(2)
memory usage: 372.0 bytes


In [20]:
df.head()

,review_id,review_text,sentiment
0,1,"The product quality is exceptional, exceeded a...",positive
1,2,Terrible customer service and the item arrived...,negative
2,3,"Average performance, decent for the price poin...",neutral
3,4,"Absolutely love this! Easy to setup, robust, a...",positive
4,5,Stopped working after two days. Extremely disa...,negative


#### Text Preprocessing and Tokenization:

Define a cleaning pipeline to strip HTML tags, remove punctuation and numbers, convert text to lowercase, remove stopwords, and apply lemmatization to reduce words to their base form.

In [24]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_and_tokenize(text):
    #Remove HTML tags
    text = re.sub(r"<.*?>", "", text)
    
    #Remove non-alphabetical characters (puntuation, digits)
    text = re.sub(r"[^a-zA-Z]", " ", text)

    #Lowercase and split into tokens
    tokens = text.lower().split()

    #Remove stopwords and lemmatize tokens longer than 2 characters
    cleaned_tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token not in stop_words and len(token) > 2
    ]
    return " ".join(cleaned_tokens)

#Apply preprocessing across the entire dataset
df["clean_text"] = df["review_text"].apply(clean_and_tokenize)

#Train-test split (using stratify to maintain class distribution across splits)
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["sentiment"],
    test_size=0.2,
    random_state=42
    #stratify=df["sentiment"] # Important for multi-class balance
)

#Feature Extraction
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

#### Train Multinomial Naive Bayes for Multi-Class Sentiment

MultinomialNB naturally supports multi-class classification out of the box using a softmax/multinomial likelihood formulation.

In [25]:
# Training Multinomial Naive Bayes on the full dataset
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
print("Model successfully trained!")

Model successfully trained!


#### Create a predict_sentiment() function that tests live custom user input phrases.

In [28]:
# Live Prediction Function
def predict_sentiment(text, model=nb_model, vectorizer=vectorizer):
    cleaned = clean_and_tokenize(text)
    vectorized = vectorizer.transform([cleaned])

    prediction = model.predict(vectorized)[0]
    probabilities = model.predict_proba(vectorized)[0]
    classes = model.classes_

    prob_dict = {cls: f"{prob * 100:.2f}%" for cls, prob in zip(classes, probabilities)}
    confidence = max(probabilities)

    return{
        "sentiment": prediction,
        "confidence": f"{confidence * 100:2f}%",
        "all_probabilities": prob_dict
    }

# Test live custom phrases
test_phrases = y_test

for phrase in test_phrases:
    result = predict_sentiment(phrase)
    print(f"Review: \"{phrase}\"")
    print(f"Predicted Sentiment: {result["sentiment"].upper()} ({result["confidence"]})")
    print(f"Breakdown: {result["all_probabilities"]}\n")

Review: "negative"
Predicted Sentiment: POSITIVE (62.500000%)
Breakdown: {np.str_('negative'): '25.00%', np.str_('neutral'): '12.50%', np.str_('positive'): '62.50%'}

Review: "negative"
Predicted Sentiment: POSITIVE (62.500000%)
Breakdown: {np.str_('negative'): '25.00%', np.str_('neutral'): '12.50%', np.str_('positive'): '62.50%'}



## The Prediction always return a positive result due to the inefficient trainig data, which is a dataset with only 10 rows